# Properties of visits and of visit sets: demo

Concept: `overview/concepts/Visit Mappings.md` (outside this repo), sections "Visits" (excursions), "Properties", "Sets of visits" and RQ4 in "Setting".
Code: `ocelescope_module_location_based_ocpm.visits` (`sequence.py`, `properties.py`, `excursions.py`); this notebook only imports and calls its functions.
Further detail: the docstrings and `docs/resources/visit-properties.md`.

Every function here works for **any set of visits**: the maximal visits $\mathcal{V}_{max}$, the visits of a mapping set, or a hand-written table.

What the notebook does:
1. loads the running example of the concept note,
2. shows the general sequence helpers (positions, runs, intervals) the visit code is built on,
3. builds the visits table of any visit set (`visit_table`),
4. derives the excursions,
5. computes the properties of every visit (kind, trivial, interrupted, contained, overlapping, counts),
6. computes the set properties (disjoint, hl-disjoint, covering) and the set metrics, for $\mathcal{V}_{max}$ (the example of the note) and for a mapping set,
7. checks RQ4.

Start Jupyter from the repository root with

```
uv run --with jupyter jupyter lab notebooks/visit_properties.ipynb
```

Conventions in all frames: one row per hu-event relation $(e, hu)$ unless stated otherwise; `null` is $\bot$; durations are polars durations (e.g. `10h`).
All functions take and return polars frames; they work on eager (`DataFrame`) and lazy (`LazyFrame`) frames alike.

In [ ]:
from pathlib import Path

import polars as pl

from ocelescope_module_location_based_ocpm.derivation.level_files import MAPPING_SCHEMA
from ocelescope_module_location_based_ocpm.visit_mapping.pipeline import derive_mapping_set
from ocelescope_module_location_based_ocpm.visits.context import location_context
from ocelescope_module_location_based_ocpm.visits.excursions import excursion_counts, excursions
from ocelescope_module_location_based_ocpm.visits.maximal import derive_maximal_visits, visit_relations, visit_table
from ocelescope_module_location_based_ocpm.visits.properties import (
    flag_rq4,
    rq4_violations,
    set_metrics,
    set_properties,
    visit_pairs,
    visit_properties,
    with_positions,
)
from ocelescope_module_location_based_ocpm.visits.sequence import (
    add_positions,
    at_positions,
    expand_intervals,
    interval_pairs,
    run_ids,
    runs,
)
from ocelescope_module_location_based_ocpm.visits.shifts import add_stops

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "mapping_example"

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250)
pl.Config.set_fmt_table_cell_list_len(8)
pl.Config.set_fmt_str_lengths(40);

## 1. Data

The running example of the concept note (`data/mapping_example/`, generated by `make_mapping_example.py`): 10 handling units, 41 events, 54 relations, Level 3; $agg$: A1, A2 → A, B1 → B, T unmapped; $V = \{T\}$, $W = \{B1\}$; $\kappa_{max} = 3$ as in the note.

In [ ]:
KAPPA = 3
relations = pl.read_csv(DATA / "relations.csv", schema=MAPPING_SCHEMA)
context = location_context(pl.read_csv(DATA / "locations.csv"))
P = pl.read_csv(DATA / "processing_events.csv")["event"].to_list()
rel, maximal = derive_maximal_visits(relations, "loc", context.agg, KAPPA, processing=P)
maximal.select("visit_id", "hu", "high_level", "first_event", "last_event", "n_relations", "n_at_location")

## 2. General sequence helpers (`sequence.py`)

The visit code works on sequences: a trace is a sequence of relations, a visit an interval of it, an excursion a run inside a visit, a visit sequence a sequence of visits.
The same few helpers do this everywhere:

| helper | what it does | used by |
|---|---|---|
| `add_positions(frame, group, order_by, name)` | position 1, 2, ... in each sequence | trace positions, stop numbers, visit sequence index |
| `run_ids(value, group, order_by)`, `runs(frame, value, group, position, keep, aggregations)` | maximal runs of equal values | excursions, high-level runs of the trace figure |
| `expand_intervals(frame, first, last, name)` | one row per position of an interval | visit relations, transfer relations |
| `at_positions(frame, sequence, position, columns, group, offset)` | columns at a position (+ offset) | entry / last / exit event of visits, dwell times |
| `interval_pairs(intervals, group, key, first, last)` | contained / overlapping pairs | pairs of visits |

Example: the runs of high-level locations of b1 (null = ⊥ is a value of its own).

In [ ]:
b1 = rel.filter(pl.col("hu") == "b1").select("hu", "position", "event", "high_level")
display(b1.with_columns(run_ids("high_level").alias("run")))
runs(b1, "high_level", aggregations=[pl.col("event").first().alias("first_event")])

In [ ]:
# Intervals of one sequence: 1 = [1, 5], 2 = [2, 3] (contained in 1), 3 = [4, 7] (overlaps 1), 4 = [8, 9] (disjoint)
intervals = pl.DataFrame({"hu": ["a"] * 4, "visit_id": [1, 2, 3, 4], "first_position": [1, 2, 4, 8], "last_position": [5, 3, 7, 9]})
interval_pairs(intervals)

## 3. The visits table of any visit set: `visit_table(intervals, relations)`

From hu, high_level and first/last position (optionally visit_id and further columns) it computes the same columns as for the maximal visits: first, last and exit event, entry/last/exit time, ends_trace, duration, $|\sigma|$, $|R_{hl}(\nu)|$, interrupted.
`maximal_visits` and the mapping sets use it, so every visit set has the same columns.
A hand-written set can be given by its first and last event; `with_positions(visits, relations)` adds the positions.

In [ ]:
hand = pl.DataFrame({"hu": ["b1", "b1", "b1"], "high_level": ["A", "B", "T"], "first_event": ["e3", "e7", "e13"], "last_event": ["e5", "e7", "e14"]})
hand = with_positions(hand, rel)
visit_table(hand.select("visit_id", "hu", "high_level", "first_position", "last_position"), rel)

## 4. Excursions: `excursions(visit_relations)`, `excursion_counts(visits, excursions)`

An excursion of $\nu$ is a maximal run of non-conforming relations strictly inside $\sigma$; its excursion locations are the high-level locations in it.
Input: the visit relations (`visit_relations(relations, visits)`, one row per visit and relation, with `at_location`).
The sweep of the mapping strategies reads the excursions of the maximal visits from this table.

In [ ]:
vrel = add_stops(visit_relations(rel, maximal), context.vehicles)
found = excursions(vrel)
found

As in the note: $m_2$ = (b1, A, e3–e11) has one excursion ⟨e7, e8⟩ with locations {B, T}; (b6, A, e26–e30) one of length 3 with {B, T}; (b4, B, e15–e21) two, ⟨e18⟩ and ⟨e20⟩.

## 5. Properties of every visit: `visit_properties(visits, visit_relations, excursions)`

| column | meaning |
|---|---|
| `kind` | ⊥ visit, stay (has a stop), vehicle visit ($L_{\neg v} = \emptyset$), transferring visit (no stop, $L_{\neg v} \neq \emptyset$) |
| `is_trivial` | $|\sigma| = 1$ |
| `interrupted` | $R(\nu) \neq R_{hl}(\nu)$ |
| `n_not_at_location`, `n_vehicle_relations`, `n_processing_at_location`, `n_excursions` | counts |
| `is_contained`, `contains`, `is_overlapping` | relative to the given set (`visit_pairs`) |

The processing count needs `is_processing_event` on the visit relations (here not added, so 0); `analyse_visit_set` adds it.

In [ ]:
props = visit_properties(maximal, vrel, found)
props.select("visit_id", "hu", "high_level", "first_event", "last_event", "kind", "is_trivial", "interrupted", "n_excursions", "n_vehicle_relations", "is_contained", "contains", "is_overlapping")

In [ ]:
visit_pairs(maximal)

## 6. Sets of visits: `set_properties(visits, visit_relations, relations)`, `set_metrics(visits, visit_relations, relations, by_hu=False)`

Per handling unit: disjoint, hl-disjoint, covering. For the set: $N$, $p_{cov}$, $N_{int}$/$p_{int}$, $N_{con}$, $N_{ov}$, $N_{dis}$/$p_{dis}$, $N_{>1}$/$p_{>1}$ (`n_in_several_visits`, `p_several`), $N_{nc}$/$p_{nc}$, $p_{\Vert}$ (`p_overlap`).

The example of the note ($\mathcal{V}_{max}$, $\kappa_{max} = 3$): $N = 24$, $p_{cov} = 1$, $N_{int} = 12$, $N_{con} = 9$, $N_{ov} = 6$, $N_{dis} = 3$ (s1, b3, b5), $p_{dis} = 3/10$, $N_{>1} = N_{nc} = 21$, $p_{\Vert} = 11/54$.

In [ ]:
display(set_properties(maximal, vrel, rel))
set_metrics(maximal, vrel, rel)

In [ ]:
set_metrics(maximal, vrel, rel, by_hu=True)

The same functions on the S6 mapping set: disjoint and covering for every handling unit.

In [ ]:
s6 = derive_mapping_set(relations, "loc", context.agg, context.vehicles, KAPPA, P, "S6", context.storage_units)
s6_vrel = add_stops(visit_relations(s6.relations, s6.mapping), context.vehicles)
display(set_properties(s6.mapping, s6_vrel, s6.relations))
set_metrics(s6.mapping, s6_vrel, s6.relations)

## 7. RQ4: `rq4_violations(relations, column="high_level")`, `flag_rq4(relations)`

All relations of a handling unit with the same timestamp should have the same (high-level) location. It holds at Levels 1 to 3; at Level 0 tied relations can differ in being located or not.

In [ ]:
print("RQ4 holds:", flag_rq4(rel))
tied = rel.filter(pl.col("event").is_in(["e3", "e4"]) & (pl.col("hu") == "b1")).with_columns(pl.col("time").min(), pl.when(pl.col("event") == "e4").then(None).otherwise("high_level").alias("high_level"))
rq4_violations(tied)